# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² protein abundance and modification dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant JSON-LD file URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset summary
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their fields and IDs, and print data samples. Entities are referenced by their `@id`.

In [ ]:
# Display available record sets by their @id
print("Available Record Sets:")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f" - @id: {rs.id} | name: {rs.name}")
    if hasattr(rs, 'fields'):
        print("   Fields (@id, name, dataType):")
        for field in rs.fields:
            print(f"     - {field.id}, {field.name}, {getattr(field, 'data_type', None)}")

# Peek at the first record of each record set
print("\nSample records from each record set:")
for rs in record_sets:
    print(f"\nRecord set {rs.id} sample record:")
    rec_iter = dataset.records(record_set=rs.id)
    try:
        print(next(rec_iter))
    except StopIteration:
        print("  No records found.")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis.

Use record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set into a dictionary of DataFrames, indexed by record set @id
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} records from record set @id: {rs.id}")
    else:
        print(f"No data found for record set @id: {rs.id}")

# For illustration, select the main protein abundance record set (by likely convention, the main table is the one with the most fields and records)
if dataframes:
    main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[1])  # Largest column count
    print(f"\nUsing main record set @id: {main_rs_id}")
    print(f"Fields (@id): {list(dataframes[main_rs_id].columns)}")
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps—filtering, normalization, grouping—on relevant numeric fields using their field `@id`s.

In [ ]:
# For EDA, choose a representative numeric field (by @id) present in the main data table
# You may need to adjust the field_id based on the printed list of columns.
df = dataframes[main_rs_id]
numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
print(f"Numeric candidates in {main_rs_id}: {numeric_candidates}")

# Example: try some common field names for proteins such as 'coverage', 'peptide_count', 'MW', etc.
for candidate in ['coverage', 'peptide_count', 'MW', 'pI', 'abundance', 'Normalized_Abundance']:
    if candidate in df.columns:
        numeric_field = candidate
        print(f"Using numeric field: {numeric_field}")
        break
else:
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"No protein-specific candidate found; using: {numeric_field}")
    else:
        raise Exception("No numeric field found for EDA.")

# Filter records for which value > 10 (arbitrary threshold for illustration)
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
group_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(grouped_df.head())

## 5. Visualization
Visualize distributions of the main numeric field and relationships between fields.

Below, we plot the histogram and boxplot for the main numeric field, and a bar chart of grouped means if grouping was performed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
plt.xlabel(numeric_field)

plt.subplot(1, 2, 2)
sns.boxplot(x=filtered_df[numeric_field])
plt.title(f"Boxplot of {numeric_field} (filtered > {threshold})")
plt.show()

# If data was grouped, plot the mean per group
if 'grouped_df' in locals():
    grouped_df.reset_index(inplace=True)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² protein abundance dataset using `mlcroissant`.

- The dataset was loaded from its Croissant schema URL with full metadata introspection through `@id`-referenced entities.
- Main record sets and their fields/columns were identified and loaded as DataFrames.
- Key numeric attributes were filtered, normalized, and visualized: this enables insight into distributions and the possibility of comparing values across biological or experimental groups.

This workflow demonstrates how to discover, extract, and begin analyzing any Croissant-described dataset with transparent, schema-aligned code referencing.